In [ ]:
# Trains RT-DETRv2 (R18) for child detection.

In [ ]:
%cd /kaggle/working

!git clone https://github.com/lyuwenyu/RT-DETR.git

%cd /kaggle/working/RT-DETR/rtdetrv2_pytorch

!pip install -r requirements.txt

In [ ]:
import yaml
from pathlib import Path

NEW_DATA_YAML = Path("/kaggle/working/data_fixed.yaml")

data_fixed = {
    "train": "/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/train/images",
    "val": "/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/valid/images",
    "test": "/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/test/images",
    "nc": 1,
    "names": ["child"]
}

with open(NEW_DATA_YAML, "w") as f:
    yaml.dump(data_fixed, f, sort_keys=False)

print("Nouveau data.yaml créé :")
print()

with open(NEW_DATA_YAML, "r") as f:
    print(f.read())

In [ ]:
from pathlib import Path

LABEL_DIRS = [
    Path("/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/train/labels"),
    Path("/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/valid/labels"),
]

all_classes = set()
bad_files = []

for label_dir in LABEL_DIRS:

    for txt_file in label_dir.glob("*.txt"):

        with open(txt_file, "r") as f:
            lines = f.readlines()

        for line_idx, line in enumerate(lines):

            parts = line.strip().split()

            if len(parts) != 5:
                bad_files.append((txt_file.name, "wrong_format"))
                continue

            try:
                cls = int(float(parts[0]))

                x, y, w, h = map(float, parts[1:])

                all_classes.add(cls)

                if cls != 0:
                    bad_files.append((txt_file.name, f"wrong_class_{cls}"))

                if not (0 <= x <= 1):
                    bad_files.append((txt_file.name, "x_outside"))

                if not (0 <= y <= 1):
                    bad_files.append((txt_file.name, "y_outside"))

                if not (0 <= w <= 1):
                    bad_files.append((txt_file.name, "w_outside"))

                if not (0 <= h <= 1):
                    bad_files.append((txt_file.name, "h_outside"))

            except:
                bad_files.append((txt_file.name, "parse_error"))

print("Classes trouvées :", sorted(all_classes))
print("Nombre de problèmes :", len(bad_files))

print()
print("Exemples :")

for bf in bad_files[:30]:
    print(bf)

In [ ]:
import json
import yaml
from pathlib import Path
from PIL import Image

DATASET_ROOT = Path("/kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset")
DATA_YAML = Path("/kaggle/working/data_fixed.yaml")

OUTPUT_ROOT = Path("/kaggle/working/RT-DETR/rtdetrv2_pytorch/dataset/child")
ANNOTATION_DIR = OUTPUT_ROOT / "annotations"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_YAML, "r") as f:
    data_yaml = yaml.safe_load(f)

names = data_yaml["names"]

categories = [
    {
        "id": i,
        "name": name,
        "supercategory": "none"
    }
    for i, name in enumerate(names)
]

def yolo_to_coco(split):

    img_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    images = []
    annotations = []

    image_id = 1
    annotation_id = 1

    for img_path in sorted(img_dir.glob("*")):

        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        with Image.open(img_path) as img:
            width, height = img.size

        images.append({
            "id": image_id,
            "file_name": img_path.name,
            "width": width,
            "height": height
        })

        label_path = label_dir / f"{img_path.stem}.txt"

        if label_path.exists():

            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:

                class_id, x_center, y_center, box_w, box_h = map(float, line.strip().split())

                x_min = (x_center - box_w / 2) * width
                y_min = (y_center - box_h / 2) * height
                bbox_w = box_w * width
                bbox_h = box_h * height

                annotations.append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(class_id),
                    "bbox": [
                        float(x_min),
                        float(y_min),
                        float(bbox_w),
                        float(bbox_h)
                    ],
                    "area": float(bbox_w * bbox_h),
                    "iscrowd": 0
                })

                annotation_id += 1

        image_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

    output_json = ANNOTATION_DIR / f"instances_{split}.json"

    with open(output_json, "w") as f:
        json.dump(coco, f)

    print(split, "terminé")
    print("Images :", len(images))
    print("Annotations :", len(annotations))

yolo_to_coco("train")
yolo_to_coco("valid")

In [ ]:
from pathlib import Path

CONFIG_DATASET_PATH = Path(
    "/kaggle/working/RT-DETR/rtdetrv2_pytorch/configs/dataset/child_detection.yml"
)

dataset_config = """
task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: 1
remap_mscoco_category: False

train_dataloader:
  type: DataLoader

  dataset:
    type: CocoDetection
    img_folder: /kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/train/images
    ann_file: /kaggle/working/RT-DETR/rtdetrv2_pytorch/dataset/child/annotations/instances_train.json
    return_masks: False

    transforms:
      type: Compose
      ops: ~

  shuffle: True
  num_workers: 2
  drop_last: True

  collate_fn:
    type: BatchImageCollateFunction

  total_batch_size: 8


val_dataloader:
  type: DataLoader

  dataset:
    type: CocoDetection
    img_folder: /kaggle/input/datasets/nisrineamalkhnifass/childdetection/full_dataset/full_dataset/valid/images
    ann_file: /kaggle/working/RT-DETR/rtdetrv2_pytorch/dataset/child/annotations/instances_valid.json
    return_masks: False

    transforms:
      type: Compose
      ops: ~

  shuffle: False
  num_workers: 2
  drop_last: False

  collate_fn:
    type: BatchImageCollateFunction

  total_batch_size: 8
"""

with open(CONFIG_DATASET_PATH, "w") as f:
    f.write(dataset_config)

print("Batch size R18 configuré")

In [ ]:
!cat /kaggle/working/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r18vd_child.yml | head -40

In [ ]:
from pathlib import Path

BASE_CONFIG = Path(
    "/kaggle/working/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r18vd_120e_coco.yml"
)

NEW_CONFIG = Path(
    "/kaggle/working/RT-DETR/rtdetrv2_pytorch/configs/rtdetrv2/rtdetrv2_r18vd_child.yml"
)

with open(BASE_CONFIG, "r") as f:
    txt = f.read()

txt = txt.replace(
    "../dataset/coco_detection.yml",
    "../dataset/child_detection.yml"
)

txt = txt.replace(
    "output_dir: ./output/rtdetrv2_r18vd_120e_coco",
    "output_dir: ./output/rtdetrv2_r18vd_child"
)

txt = txt.replace(
    "epoches: 120",
    "epoches: 100"
)

with open(NEW_CONFIG, "w") as f:
    f.write(txt)

print("Config R18 créée")

In [ ]:
%cd /kaggle/working/RT-DETR/rtdetrv2_pytorch

!python tools/train.py \
  -c configs/rtdetrv2/rtdetrv2_r18vd_child.yml \
  --use-amp \
  --seed=0

In [ ]:
import os
import shutil

# Supprimer les checkpoints intermédiaires (garder seulement le best)
output_dir = '/kaggle/working/RT-DETR/rtdetrv2_pytorch/output/rtdetrv2_r18vd_child'

for f in os.listdir(output_dir):
    if f.startswith('checkpoint') and f != 'checkpoint_best.pth':
        os.remove(os.path.join(output_dir, f))
        print(f"Supprimé : {f}")

print("Espace libéré !")